# Modélisation générative de C-Haines — copie corrigée

Copie de `Kelian2909/generative_ia`, commit `bca170c92297e2155c28663772c2bd9914987e1f`.

Le fichier contient **31 jours de juillet 2023**. Ce notebook compare des distributions scalaires sur un découpage chronologique ; il ne valide ni un modèle annuel, ni des champs atmosphériques cohérents, ni une tarification de feux. Les anciens résultats et conclusions de production ont été retirés car ils reposaient sur un autre indicateur et une évaluation sur les données d’apprentissage.


In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / "config.py").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import config
config.set_seed()


## 1. Indicateur et qualité des données

Le C-Haines utilise $C_A=(T_{850}-T_{700})/2-2$ et $C_B=(T_{850}-T_{d,850})/3-1$, avec les corrections de sécheresse de Mills et McCaw (2010). Le point de rosée est approché par Magnus. Les humidités hors $]0,100]$ sont masquées et comptées, sans les tronquer en observations valides. `method="legacy_proxy"` conserve l’ancien indicateur pour comparaison uniquement.


In [ ]:
from src.data_loader import load_haines
from src.preprocessing import build_dataframe
ds, index = load_haines(config.DATA_PATH)
df = build_dataframe(index)
print(index.attrs)
print(df.haines_index.describe())


## 2. Séparation par dates et transformation train uniquement

Toutes les mailles d’une date restent dans le même échantillon. Deux dates sont exclues aux frontières, pour créer un jour d’embargo. La standardisation ne borne pas les sorties au minimum et au maximum appris ; cela autorise l’extrapolation numérique sans garantir sa plausibilité physique.


In [ ]:
from src.preprocessing import temporal_split, fit_transform, transform
train_raw, validation_raw, test_raw = temporal_split(df, gap_days=1)
train, lmbda, scaler, shift = fit_transform(train_raw)
validation = transform(validation_raw, scaler)
test = transform(test_raw, scaler)
for label, part in [("train", train), ("validation", validation), ("test", test)]:
    print(label, len(part), part.valid_time.min(), part.valid_time.max())


## 3. Modèles et objectifs

- **VAE** : vraisemblance gaussienne apprise, sortie non bornée et bruit d’observation lors de la génération. Le terme KL reste explicite.
- **VAE pondéré** : modification bornée de l’objectif de reconstruction ; ce n’est pas une preuve de calibration de la queue.
- **GAN/cGAN** : discriminateur à logits et `BCEWithLogitsLoss`, avec cibles standardisées. Conditions : latitude, longitude et mois cyclique. Un seul mois est observé : aucune validation saisonnière.
- **WGAN-GP** : perte $E[D(G(z))]-E[D(X)]+\lambda E[(\|
abla D(\hat X)\|_2-1)^2]$ pour le critique ; perte $-E[D(G(z))]$ pour le générateur. Pas de pénalité artificielle qui augmente la variance pour créer des extrêmes.


In [ ]:
# Reproduire le benchmark complet : décommenter si souhaité.
# from src.run_experiment import run_experiment
# report = run_experiment(epochs=10, n_samples=10000, seed=100)
report = json.loads((ROOT / "reports/benchmark.json").read_text())
import pandas as pd
rows = [{"model": name, "split": split, **values[split]}
        for name, values in report["models"].items() for split in ["validation", "test"]]
scores = pd.DataFrame(rows)
scores[["model", "split", "wasserstein", "ks_stat", "q99_real", "q99_gen", "std_gen"]]


## 4. Comparaison avec des références simples

L’échantillonnage empirique de train et une loi normale ajustée sur train sont comparés aux réseaux. Un réseau ne doit pas être déclaré meilleur sur sa seule architecture ou sur son maximum simulé. Les p-values KS supposent des observations indépendantes ; ici elles restent des diagnostics sans interprétation de significativité générale.


In [ ]:
ax = scores.pivot(index="model", columns="split", values="wasserstein").plot.bar(
    figsize=(10, 4), title="Distance de Wasserstein — plus faible = meilleur")
ax.set_ylabel("Unités C-Haines")


## 5. Paiement paramétrique : unité d’observation explicite

Pour une indemnité $L$ et un seuil $u$, $E[P]=L\,P(H>u)$. Le seuil livré est le quantile 95 % de train, **un seuil de démonstration**, sans calibration contractuelle. L’unité est une maille et un pas de temps ; la transformation en prime annuelle exigerait un portefeuille, des garanties et une loi jointe temporelle/spatiale.


In [ ]:
print("Seuil de démonstration :", report["trigger_demo"])
print("Unité :", report["payment_unit"])
scores[["model", "split", "trigger_probability", "observed_trigger_probability", "expected_payment_per_observation"]]


## Limites et suite du travail

Le test couvre seulement les derniers jours du même mois, sur les mêmes localisations. Un modèle scalaire ne reconstitue pas la dépendance d’un champ météorologique. Les niveaux 850/700 hPa peuvent être sous le relief ; aucun masque orographique n’est disponible ici. Le lien avec des départs de feux et des pertes n’est pas observé. Une étude climatique nécessite plusieurs années, plusieurs saisons, une validation géographique et des données de sinistres/exposition.

Les échecs de benchmark et les essais de validation sont conservés dans `reports/`. Consulter `reports/AUDIT.md` et `reports/RESULTATS.md` avant de tirer des conclusions.
